# 01. データ確認 + EDA: YANS-official/ogiri-bokete (text_to_text)

- データ取得・整形ロジックは `src/data.py` に集約
- このnotebookは EDA 用 (HF から取得して中身を眺めるだけ)

In [ ]:
from pathlib import Path
import sys

if "google.colab" not in sys.modules:
    try:
        get_ipython().run_line_magic("load_ext", "autoreload")
        get_ipython().run_line_magic("autoreload", "2")
    except Exception as e:
        print("autoreload skipped:", e)

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401

from src.data import fetch_oogiri_t2t
ROOT

## データ取得

In [ ]:
df = fetch_oogiri_t2t()
print("rows:", len(df))
display(df.head())

## EDA

In [ ]:
display(df.isna().sum().to_frame("na_count"))

In [ ]:
df["odai_len"] = df["odai"].fillna("").str.len()
df["text_len"] = df["text"].fillna("").str.len()
display(df[["odai_len", "text_len", "score"]].describe())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df["odai_len"].hist(bins=30, ax=axes[0]); axes[0].set_title("お題の文字数")
df["text_len"].hist(bins=30, ax=axes[1]); axes[1].set_title("回答の文字数")
df["score"].clip(upper=df["score"].quantile(0.99)).hist(bins=30, ax=axes[2])
axes[2].set_title("score分布 (上位1%クリップ)")
plt.tight_layout()
plt.show()

In [ ]:
# お題ごとの回答数 (preferenceペア構築の見積もり用)
per_odai = df.groupby("odai_id").size().rename("n_responses").reset_index()
print("ユニークお題数:", len(per_odai))
display(per_odai["n_responses"].describe().to_frame())

In [ ]:
# 高scoreの例を眺める
display(df.sort_values("score", ascending=False)[["odai", "text", "score"]].head())